In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor

from joblib import dump, load
import itertools

In [3]:
transcriptomics = load("data/transcriptomics_matched.joblib")
proteomics = load("data/proteomics_matched.joblib")

In [4]:
indices = proteomics.index

In [ ]:
# OLD ONE

def boosting_pipeline(model, param_grid, X, y, anova_features):

    # get samples and proteins
    sample_list = y.index
    protein_list = y.columns

    # initialize the results table
    df = pd.DataFrame(columns=['protein',
                               'LOO sample',
                               'true',
                               'predicted',
                               'params'])

    for sample in sample_list:
        print(sample)

        # leave one out split
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1)

        # get folds
        kf = KFold(n_splits=5, shuffle=True)
        folds = list(kf.split(X_train.index))

        # get parameter combinations
        keys, values = zip(*param_grid.items())
        param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

        for protein in protein_list:
            
            # remove all proteins but one from y
            yp_train = y_train[protein]
            yp_test = y_test[protein]

            # select the best features by F-value
            selector = SelectKBest(score_func=f_regression, k=anova_features)
            selector.fit_transform(X_train, yp_train)
            
            # remove all other features from X
            selected_indices = np.where(selector.get_support())[0]
            selected_features = X_train.columns[selected_indices]
            Xp_train = X_train[selected_features]
            Xp_test = X_test[selected_features]

            # initialize best_params variable
            best_params = param_combinations[0]
            best_params_mse = np.inf

            for params in param_combinations:

                avg_mse = 0

                for fold in folds:

                    # fit and validation indices
                    fit_idx, val_idx = fold

                    # construct sets
                    X_fit = Xp_train.iloc[fit_idx]
                    y_fit = yp_train.iloc[fit_idx]
                    X_val = Xp_train.iloc[val_idx]
                    y_val = yp_train.iloc[val_idx]

                    # build model for this fold and get predictions
                    estimator = model
                    estimator.set_params(**params)

                    estimator.fit(X_fit, y_fit)
                    y_pred = estimator.predict(X_val)

                    # get mse on this fold
                    mse = mean_squared_error(y_val, y_pred)
                    avg_mse += mse
                
                # get mse for parameter configuration
                avg_mse /= len(folds)

                # update best config if these parameters have the lowest mean mse
                if avg_mse < best_params_mse:
                    best_params_mse = avg_mse
                    best_params = params
                
            # train the final model with the best parameters
            estimator = model
            estimator.set_params(**best_params)

            estimator.fit(Xp_train, yp_train)
            y_pred = model.predict(Xp_test)

            # add entry to the results table
            df.loc[len(df)] = [
                protein,
                sample,
                yp_test.values[0],
                y_pred[0],
                best_params
            ]
        
    return df

In [31]:
def full_adaptive_pipeline(model, param_grid, X, y, anova_features):

    # get samples and proteins
    sample_list = y.index[:3]
    protein_list = y.columns[:2]

    # initialize the results table
    df = pd.DataFrame(columns=['protein',
                               'LOO sample',
                               'true',
                               'predicted',
                               'params'])
    
    # get parameter combinations
    keys, values = zip(*param_grid.items())
    param_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]
    del keys, values
    
    # nested loops
    for protein in protein_list:
            
        # remove all proteins but one from y
        yp = y[protein]

        for sample in sample_list:

            # leave one out split
            X_train, X_test, y_train, y_test = train_test_split(X, yp, test_size=1)

            # get folds
            kf = KFold(n_splits=5, shuffle=True)
            folds = list(kf.split(X_train.index))

            avg_mse = [0] * len(param_combinations)
            all_features = []

            for fold in folds:
                
                # fit and validation indices
                fit_idx, val_idx = fold

                # construct sets
                X_fit = X_train.iloc[fit_idx]
                y_fit = y_train.iloc[fit_idx]
                X_val = X_train.iloc[val_idx]
                y_val = y_train.iloc[val_idx]

                # select the best features by F-value
                selector = SelectKBest(score_func=f_regression, k=anova_features)
                selector.fit_transform(X_fit, y_fit)
                
                # remove all other features from X
                selected_indices = np.where(selector.get_support())[0]
                selected_features = list(X_fit.columns[selected_indices])
                X_fit = X_fit[selected_features]
                X_val = X_val[selected_features]

                all_features.append(selected_features)

                for k in range(len(param_combinations)):

                    params = param_combinations[k]

                    # build model for this fold and get predictions
                    estimator = model
                    estimator.set_params(**params)

                    estimator.fit(X_fit, y_fit)
                    y_pred = estimator.predict(X_val)

                    # get mse on this fold
                    mse = mean_squared_error(y_val, y_pred)
                    avg_mse[k] += mse
            
            k_max = np.argmax(avg_mse)
            best_params = param_combinations[k_max]
            all_features = list(set(list(all_features)))

            # train the final model with the best parameters
            estimator = model
            estimator.set_params(**best_params)

            estimator.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            # add entry to the results table
            df.loc[len(df)] = [
                protein,
                sample,
                y_test.values[0],
                y_pred[0],
                best_params
            ]
    
    return df

## XGBoost

In [32]:
xgb = XGBRegressor(device='cpu', tree_method='hist')

In [33]:
param_grid = {
    'eta' : [0.03, 0.3],
    'max_depth' : [3, 6],
    'subsample' : [0.5, 1]
}

In [34]:
xgb_scores = full_adaptive_pipeline(
    model = xgb,
    param_grid = param_grid,
    X = transcriptomics,
    y = proteomics,
    anova_features = 20
)

TypeError: unhashable type: 'list'

In [16]:
xgb_scores

,protein,LOO sample,true,predicted,params
0,A2M,MDAMB468_BREAST,0.095513,-1.005652,"{'eta': 0.05, 'max_depth': 3}"
1,A2M,SH4_SKIN,0.094028,0.226838,"{'eta': 0.05, 'max_depth': 3}"
2,A2M,AU565_BREAST,1.250857,0.147475,"{'eta': 0.05, 'max_depth': 3}"
3,A2M,KMRC1_KIDNEY,-1.680788,-1.203801,"{'eta': 0.05, 'max_depth': 3}"
4,A2M,CAL51_BREAST,0.468455,1.458798,"{'eta': 0.05, 'max_depth': 3}"
